# A/B Testing — KKBox Retention Campaign
**File:** `05_AB_Testing.ipynb`
**Inputs:** `master_model_table.parquet`, `retention_decision_table.parquet`, `behavior_retention_weights.csv`
**Outputs:** `ab_tracking_table.csv`, `ab_backtest_results.csv`, `ab_test_metadata.json`

---

## Why we use February 2017 as a backtest

The March 2017 inference population has membership expiry dates in **April 2017**.
Observing renewal outcomes requires April/May 2017 transaction data, which is absent
from the public dataset (confirmed: 0 rows with `transaction_date >= 2017-04-01`,
despite 1,025,980 memberships expiring in April).

The February 2017 population (970,960 users) has official churn labels in `train_v2.csv`.
We apply the A/B framework retrospectively: assign arms, apply voucher logic, then
observe actual `is_churn` outcomes as ground truth.

## Critical design rule: backtest on TARGETED users only

The A/B test must be run **only on non-Stable users** (those who would actually
receive a voucher). Including Stable users (99.99% natural renewal) in all arms
dilutes the signal and produces a spurious null result — not because the voucher
does nothing, but because 35% of every arm consists of users who always renew.

## What we are verifying

| Question | Metric |
|---|---|
| Does the voucher lift renewal? | Incremental renewal rate: Treatment vs Control (non-Stable only) |
| Are save_rate assumptions valid? | Control arm natural renewal by tier vs assumed save_rate targets |
| Tier-based vs flat voucher? | T\_A (5/10/20%) vs T\_B (flat 10%) renewal rates |
| Which tier responds most? | Stratified renewal lift by risk tier |

## Three arms (non-Stable population only)

| Arm | Share | Treatment |
|---|---|---|
| Control (C) | 20% | No voucher — observe natural renewal |
| Treatment A (T\_A) | 40% | Tier-based voucher (5 / 10 / 20%) |
| Treatment B (T\_B) | 40% | Flat 10% voucher for all tiers |


In [84]:
# ===== Imports =====
from pathlib import Path
import json
import datetime
import hashlib
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.stats import norm
from statsmodels.stats.proportion import proportions_ztest

DATA_DIR = Path("Data")

CAMPAIGN_ID     = "campaign_2017_02_retention_backtest"
SIM_CAMPAIGN_ID = "sim_march_2017"   # separate ID for simulation arm assignment
SEND_DATE     = datetime.date(2017, 2, 1)
MDE           = 0.02    # minimum detectable effect: 2 percentage points
ALPHA         = 0.05
POWER         = 0.80
MAX_VOUCHER   = 200.0

print("Imports OK")
print(f"Backtest population: February 2017 (actual labels from train_v2.csv)")
print(f"MDE={MDE:.1%} | alpha={ALPHA} | power={POWER:.0%}")


Imports OK
Backtest population: February 2017 (actual labels from train_v2.csv)
MDE=2.0% | alpha=0.05 | power=80%


## Step 1: Load Data

Three inputs are merged on `msno`:
- `master_model_table.parquet` — February 2017 features + actual `is_churn` labels
- `retention_decision_table.parquet` — CLV, retention cost, voucher parameters from `04_RetentionDecision.ipynb`
- `behavior_retention_weights.csv` — data-driven behavioral weights


In [100]:
# ===== Step 1: Load Data =====
MASTER_PATH    = DATA_DIR / "master_model_table.parquet"
DECISION_PATH  = DATA_DIR / "retention_decision_table.parquet"
WEIGHTS_PATH   = DATA_DIR / "behavior_retention_weights.csv"

for p in [MASTER_PATH, DECISION_PATH, WEIGHTS_PATH]:
    print(f"{p}: {'FOUND' if p.exists() else 'MISSING'}")

master   = pd.read_parquet(MASTER_PATH)
decision = pd.read_parquet(DECISION_PATH)
weights_df = pd.read_csv(WEIGHTS_PATH)

# February 2017 population with actual labels
val = master[master["dataset_split"] == "validation"].copy()

print(f"\nFebruary 2017 population : {len(val):,}")
print(f"Actual churn rate        : {val['is_churn'].mean():.3%}")
print(f"Actual renewal rate      : {(1-val['is_churn']).mean():.3%}")

# 05-1 FIX: Do NOT merge March-derived CLV onto February users.
# CLV must be derived from February user features (their own payment history),
# not from the March retention_decision_table which covers a different population.
# We merge only p_churn predictions; all financial params are recomputed from
# February master features below.

# Get p_churn from submission (March model scores) for Feb users
# Note: p_churn was trained on Jan data and validated on Feb — valid to apply here
if "p_churn" in decision.columns:
    val = val.merge(decision[["msno","p_churn"]], on="msno", how="left")
    val["p_churn"] = val["p_churn"].fillna(val["p_churn"].median())
else:
    # Fallback: use is_churn probability = 0 if no model score available
    val["p_churn"] = 0.0
    print("Warning: p_churn not found in decision table — using 0")

# Compute CLV from February user features directly
MONTHLY_CHURN_RATE_FEB = float(val["is_churn"].mean())  # actual Feb churn rate
# Debug: check if columns exist and have values
print("avg_amount_paid in val.columns:", "avg_amount_paid" in val.columns)
print("avg_amount_paid non-zero:", (val["avg_amount_paid"] > 0).sum() if "avg_amount_paid" in val.columns else "MISSING")

plan_days_feb = val["avg_plan_days"].replace(0, 30) if "avg_plan_days" in val.columns else pd.Series(30, index=val.index)
avg_paid_feb = val["avg_amount_paid"].clip(lower=0)

nonzero_median = avg_paid_feb[avg_paid_feb > 0].median()
avg_paid_feb_filled = avg_paid_feb.replace(0, nonzero_median)

plan_days_feb = val["avg_plan_days"].replace(0, 30) if "avg_plan_days" in val.columns \
                else pd.Series(30, index=val.index)
plan_days_feb_filled = plan_days_feb

val["monthly_revenue"] = avg_paid_feb_filled / (plan_days_feb_filled / 30.0)
val["clv"] = (val["monthly_revenue"] * (1.0 / MONTHLY_CHURN_RATE_FEB)).clip(lower=0)

# Merge voucher parameters from decision table (risk_tier etc.) for arm assignment context
dec_cols = [c for c in ["msno","risk_tier","segment","voucher_pct"] if c in decision.columns]
val = val.merge(decision[dec_cols], on="msno", how="left")

print(f"CLV derived from February features:")
print(f"  Feb churn rate used   : {MONTHLY_CHURN_RATE_FEB:.3%}")
print(f"  CLV median            : ${val["clv"].median():,.0f}")
print(f"  NaN risk_tier         : {val["risk_tier"].isna().sum():,}")
# Fill NaN risk_tier for Feb users not in March decision table
if val["risk_tier"].isna().any():
    # Re-derive from p_churn quantiles for unmatched users
    Q1_feb = val["p_churn"].quantile(0.25)
    Q2_feb = val["p_churn"].quantile(0.50)
    Q3_feb = val["p_churn"].quantile(0.75)
    def _tier(p):
        if pd.isna(p) or p < Q1_feb: return "Stable"
        elif p < Q2_feb: return "Medium"
        elif p < Q3_feb: return "High"
        else: return "Critical"
    val["risk_tier"] = val.apply(
        lambda r: _tier(r["p_churn"]) if pd.isna(r["risk_tier"]) else r["risk_tier"], axis=1
    )
    print(f"  NaN risk_tier after fill: {val["risk_tier"].isna().sum():,}")

# Load behavioral weights
BEHAVIOR_WEIGHTS = dict(zip(weights_df["feature"], weights_df["weight"]))
print(f"\nBehavioral weights loaded: {len(BEHAVIOR_WEIGHTS)} features")
for k, v in BEHAVIOR_WEIGHTS.items():
    print(f"  {k:<30}: {v:+.4f}")


Data\master_model_table.parquet: FOUND
Data\retention_decision_table.parquet: FOUND
Data\behavior_retention_weights.csv: FOUND

February 2017 population : 970,960
Actual churn rate        : 8.994%
Actual renewal rate      : 91.006%
avg_amount_paid in val.columns: True
avg_amount_paid non-zero: 361381
CLV derived from February features:
  Feb churn rate used   : 8.994%
  CLV median            : $1,657
  NaN risk_tier         : 169,470
  NaN risk_tier after fill: 0

Behavioral weights loaded: 6 features
  behavior_auto_renew           : +0.1247
  behavior_cancel               : -0.2500
  behavior_has_log              : -0.0213
  behavior_active_no_cancel     : -0.2500
  behavior_high_total_secs      : -0.0211
  behavior_long_plan            : -0.2500


In [101]:
print(f"avg_amount_paid zero count: {(val['avg_amount_paid'] == 0).sum():,} ({(val['avg_amount_paid']==0).mean():.1%})")
print(f"avg_amount_paid > 0 count:  {(val['avg_amount_paid'] > 0).sum():,}")

avg_amount_paid zero count: 531,861 (54.8%)
avg_amount_paid > 0 count:  361,381


## Step 2: Sample Size Calculation

Before running the experiment, verify that the population is large enough
to detect the target effect with sufficient statistical power.

Using the two-proportion z-test formula:
$$n = \frac{(z_{\alpha/2} + z_{\beta})^2 \cdot [p_0(1-p_0) + p_1(1-p_1)]}{(p_1 - p_0)^2}$$

Where:
- $p_0$ = baseline renewal rate (control arm)
- $p_1 = p_0 + \text{MDE}$ (expected renewal rate with voucher)
- $z_{\alpha/2} = 1.96$ for $\alpha = 0.05$ two-tailed
- $z_{\beta} = 0.84$ for 80% power


In [102]:
# ===== Step 2: Sample Size Calculation =====
baseline_renewal = 1 - val["is_churn"].mean()
p0 = baseline_renewal
p1 = p0 + MDE

z_alpha = norm.ppf(1 - ALPHA / 2)   # two-tailed
z_beta  = norm.ppf(POWER)

n_per_arm = int(np.ceil(
    (z_alpha + z_beta)**2 * (p0*(1-p0) + p1*(1-p1)) / (MDE**2)
))

print("=" * 55)
print("SAMPLE SIZE ANALYSIS")
print("=" * 55)
print(f"  Baseline renewal rate (p0) : {p0:.3%}")
print(f"  Target renewal rate   (p1) : {p1:.3%}")
print(f"  Minimum detectable effect  : {MDE:.1%}")
print(f"  alpha = {ALPHA}, power = {POWER:.0%}")
print(f"  Required per arm           : {n_per_arm:,}")
print(f"  Required total (3 arms)    : {n_per_arm*3:,}")
print(f"  Available (Feb 2017 pop)   : {len(val):,}")
print(f"  Sufficient?                : {'YES' if len(val) >= n_per_arm*3 else 'NO'}")

# Non-Stable eligible users (those who would be targeted)
eligible_mask = val["risk_tier"].notna() & (val["risk_tier"] != "Stable")
n_eligible    = eligible_mask.sum()
print(f"\n  Eligible (non-Stable)      : {n_eligible:,}")
print(f"  Required eligible per arm  : {n_per_arm:,}")
print(f"  Eligible sufficient?       : {'YES' if n_eligible >= n_per_arm*3 else 'NO'}")


SAMPLE SIZE ANALYSIS
  Baseline renewal rate (p0) : 91.006%
  Target renewal rate   (p1) : 93.006%
  Minimum detectable effect  : 2.0%
  alpha = 0.05, power = 80%
  Required per arm           : 2,883
  Required total (3 arms)    : 8,649
  Available (Feb 2017 pop)   : 970,960
  Sufficient?                : YES

  Eligible (non-Stable)      : 817,388
  Required eligible per arm  : 2,883
  Eligible sufficient?       : YES


## Step 3: Randomized Arm Assignment

Users are assigned using deterministic MD5 hashing of `msno + campaign_id`.
This guarantees:
- **Reproducibility** — same user always gets the same arm across runs
- **No contamination** — assignment is purely by user identity, not behavior
- **Balance** — hash values are uniformly distributed, giving ~equal arm sizes
- **Stratification** by risk tier — ensures each tier is balanced across arms


In [103]:
# ===== Step 3: Arm Assignment (non-Stable targeted users only) =====
def assign_arm(msno, campaign_id=CAMPAIGN_ID):
    """Deterministic arm assignment via MD5 hash bucketing."""
    h = hashlib.md5(f"{campaign_id}:{msno}".encode()).hexdigest()
    bucket = int(h[:8], 16) / 0xFFFFFFFF   # uniform [0, 1]
    if bucket < 0.20:   return "C"    # control 20%
    elif bucket < 0.60: return "T_A"  # treatment A 40%
    else:               return "T_B"  # treatment B 40%

# CRITICAL: restrict to non-Stable users only
# Stable users (99.99% natural renewal) must NOT be included in any arm.
# Including them dilutes the signal — their renewal behavior is unaffected
# by vouchers and swamps the 9% churn signal we are trying to detect.
targeted = val[val["risk_tier"] != "Stable"].copy()

targeted["arm"] = targeted["msno"].apply(assign_arm)

print(f"Full February population   : {len(val):,}")
print(f"Stable (excluded from test): {(val['risk_tier']=='Stable').sum():,} ({(val['risk_tier']=='Stable').mean():.1%})")
print(f"Non-Stable (test pool)     : {len(targeted):,} ({len(targeted)/len(val):.1%})")
print()
print("Arm assignment (non-Stable only):")
arm_dist = targeted["arm"].value_counts().sort_index()
for arm, cnt in arm_dist.items():
    print(f"  {arm}: {cnt:>8,} ({cnt/len(targeted):.1%})")

print()
print("Balance check by risk tier:")
balance = targeted.groupby(["risk_tier","arm"], observed=True).size().unstack(fill_value=0)
print(balance.to_string())

# Assign voucher
FLAT_VOUCHER_PCT = 0.10

def get_voucher_pct(row):
    if row["arm"] == "C":     return 0.0
    elif row["arm"] == "T_A": return row["voucher_pct"] if pd.notna(row.get("voucher_pct")) else 0.0
    else:                     return FLAT_VOUCHER_PCT

targeted["assigned_voucher_pct"] = targeted.apply(get_voucher_pct, axis=1)
avg_paid = targeted["avg_amount_paid"].clip(lower=0) if "avg_amount_paid" in targeted.columns else pd.Series(149.0, index=targeted.index)
targeted["assigned_cost"] = (avg_paid * targeted["assigned_voucher_pct"]).clip(upper=MAX_VOUCHER)

print()
print("Voucher cost by arm:")
print(targeted.groupby("arm")["assigned_cost"].agg(["mean","sum"]).round(2).to_string())


Full February population   : 970,960
Stable (excluded from test): 153,572 (15.8%)
Non-Stable (test pool)     : 817,388 (84.2%)

Arm assignment (non-Stable only):
  C:  163,304 (20.0%)
  T_A:  326,896 (40.0%)
  T_B:  327,188 (40.0%)

Balance check by risk tier:
arm            C     T_A     T_B
risk_tier                       
Critical   66594  133930  133814
High       57104  113897  113974
Medium     39606   79069   79400

Voucher cost by arm:
     mean         sum
arm                  
C     0.0        0.00
T_A   5.6  1656503.10
T_B   6.6  1953025.33


## Step 4: Backtest Using Actual February 2017 Labels

We use the **actual observed `is_churn` labels** from `train_v2.csv` as the
outcome variable. The control arm shows the **natural renewal rate** — what
happens without any voucher intervention.

**Important distinction:**
- **Renewal rate** = fraction of users who renewed (regardless of voucher)
- **Incremental lift** = renewal_rate(treatment) − renewal_rate(control)
- Only incremental lift should be credited to the campaign

The save_rate validation compares:
- **Assumed save_rate** = what we assumed in `04_RetentionDecision.ipynb` per tier
- **Observed natural renewal** = what actually happened without intervention (control arm)

If observed natural renewal is already 85%, our voucher needs to push it to
85% + assumed_save_rate to justify the cost.


In [104]:
# ===== Step 4: Backtest Analysis (non-Stable users only) =====
targeted["renewed"] = (targeted["is_churn"] == 0).astype(int)

arm_summary = targeted.groupby("arm", observed=True).agg(
    n_users      = ("msno","count"),
    n_renewed    = ("renewed","sum"),
    renewal_rate = ("renewed","mean"),
    avg_p_churn  = ("p_churn","mean") if "p_churn" in targeted.columns else ("is_churn","mean"),
).round(4)
arm_summary["churn_rate"] = 1 - arm_summary["renewal_rate"]

print("=" * 60)
print("ARM-LEVEL RENEWAL RATES (non-Stable users only)")
print("=" * 60)
print(arm_summary.to_string())

control_renewal = arm_summary.loc["C","renewal_rate"]

print("\nIncremental lift vs Control:")
for arm in ["T_A","T_B"]:
    if arm in arm_summary.index:
        lift = arm_summary.loc[arm,"renewal_rate"] - control_renewal
        print(f"  {arm}: {lift:+.4f} ({lift:+.2%})")

print()
print("Note on result interpretation:")
print("  If lift ≈ 0 on targeted users, the voucher is genuinely ineffective.")
print("  If lift ≈ 0 only because Stable users were included, that is a design flaw.")
print("  This run correctly excludes Stable users, so the result is interpretable.")


ARM-LEVEL RENEWAL RATES (non-Stable users only)
     n_users  n_renewed  renewal_rate  avg_p_churn  churn_rate
arm                                                           
C     163304     145900        0.8934       0.0725      0.1066
T_A   326896     292011        0.8933       0.0727      0.1067
T_B   327188     292152        0.8929       0.0727      0.1071

Incremental lift vs Control:
  T_A: -0.0001 (-0.01%)
  T_B: -0.0005 (-0.05%)

Note on result interpretation:
  If lift ≈ 0 on targeted users, the voucher is genuinely ineffective.
  If lift ≈ 0 only because Stable users were included, that is a design flaw.
  This run correctly excludes Stable users, so the result is interpretable.


## Step 5: Statistical Hypothesis Tests

**Primary test:** Two-proportion z-test (one-sided, $H_1$: treatment renewal rate > control)

$$z = \frac{\hat{p}_T - \hat{p}_C}{\sqrt{\hat{p}(1-\hat{p})\left(\frac{1}{n_T} + \frac{1}{n_C}\right)}}$$

**Decision rule:**
- $p < 0.05$ → statistically significant lift → SHIP the campaign
- $p \geq 0.05$ → no evidence of lift → DO NOT SHIP or REDESIGN

**Secondary tests:**
- Treatment A vs Treatment B: which voucher structure works better?
- Stratified by risk tier: which tier benefits most?


In [105]:
# ===== Step 5: Hypothesis Tests (non-Stable users only) =====
from statsmodels.stats.proportion import proportions_ztest

results = {}

c_renewed = int(arm_summary.loc["C","n_renewed"])
c_n       = int(arm_summary.loc["C","n_users"])

print("=" * 60)
print("HYPOTHESIS TEST RESULTS (one-sided, alpha=0.05)")
print("Non-Stable targeted users only")
print("=" * 60)

for arm in ["T_A","T_B"]:
    if arm not in arm_summary.index:
        continue
    t_renewed = int(arm_summary.loc[arm,"n_renewed"])
    t_n       = int(arm_summary.loc[arm,"n_users"])

    z, p = proportions_ztest(
        count=[t_renewed, c_renewed],
        nobs=[t_n, c_n],
        alternative="larger"
    )
    lift = arm_summary.loc[arm,"renewal_rate"] - control_renewal
    results[arm] = {"z":z,"p":p,"lift":lift}

    verdict = "SIGNIFICANT" if p < ALPHA else "NOT SIGNIFICANT"
    print(f"\n{arm} vs Control:")
    print(f"  Renewal rate: {arm_summary.loc[arm,'renewal_rate']:.3%} vs {control_renewal:.3%}")
    print(f"  Lift        : {lift:+.4f} ({lift:+.2%})")
    print(f"  Z-statistic : {z:.4f}")
    print(f"  P-value     : {p:.5f}")
    print(f"  Result      : {verdict}")

if "T_A" in arm_summary.index and "T_B" in arm_summary.index:
    ta_r = int(arm_summary.loc["T_A","n_renewed"])
    ta_n = int(arm_summary.loc["T_A","n_users"])
    tb_r = int(arm_summary.loc["T_B","n_renewed"])
    tb_n = int(arm_summary.loc["T_B","n_users"])
    z_ab, p_ab = proportions_ztest([ta_r,tb_r],[ta_n,tb_n],alternative="two-sided")
    lift_ab = arm_summary.loc["T_A","renewal_rate"] - arm_summary.loc["T_B","renewal_rate"]
    print(f"\nT_A vs T_B (two-sided):")
    print(f"  Lift: {lift_ab:+.4f} | Z={z_ab:.4f} | P={p_ab:.5f}")
    winner = "T_A (tier-based) significantly better" if p_ab < ALPHA and lift_ab > 0 else "T_B (flat) significantly better" if p_ab < ALPHA else "No significant difference between voucher structures"
    print(f"  {winner}")


HYPOTHESIS TEST RESULTS (one-sided, alpha=0.05)
Non-Stable targeted users only

T_A vs Control:
  Renewal rate: 89.330% vs 89.340%
  Lift        : -0.0001 (-0.01%)
  Z-statistic : -0.1514
  P-value     : 0.56018
  Result      : NOT SIGNIFICANT

T_B vs Control:
  Renewal rate: 89.290% vs 89.340%
  Lift        : -0.0005 (-0.05%)
  Z-statistic : -0.5425
  P-value     : 0.70627
  Result      : NOT SIGNIFICANT

T_A vs T_B (two-sided):
  Lift: +0.0004 | Z=0.4793 | P=0.63169
  No significant difference between voucher structures


In [106]:
# ===== Step 6: Stratified Analysis by Risk Tier =====
print("=" * 65)
print("STRATIFIED RENEWAL RATES BY RISK TIER (non-Stable users)")
print("=" * 65)

tier_arm = targeted.groupby(["risk_tier","arm"], observed=True).agg(
    n_users      = ("msno","count"),
    n_renewed    = ("renewed","sum"),
    renewal_rate = ("renewed","mean"),
).round(4).unstack("arm")

print(tier_arm.to_string())

# Save rate validation with honest ceiling check
print()
print("=" * 65)
print("SAVE RATE VALIDATION")
print("=" * 65)
print("Comparing assumed save_rates (from 04_RetentionDecision) against")
print("natural renewal rates in control arm. If ctrl_renewal + save_rate > 100%,")
print("the assumption is physically impossible to achieve.")
print()

assumed = {"Medium":0.08, "High":0.15, "Critical":0.07}
print(f"{'Tier':<12} {'Ctrl renewal':>14} {'Assumed save':>14} {'Target':>12} {'Achievable?':>13}")
print("-"*68)
for tier, save_rate in assumed.items():
    if tier in tier_arm.index:
        ctrl_r = tier_arm.loc[tier,("renewal_rate","C")] if ("renewal_rate","C") in tier_arm.columns else 0
        target = ctrl_r + save_rate
        achievable = "YES" if target <= 1.0 else "NO -- exceeds 100%"
        print(f"{tier:<12} {ctrl_r:>14.3%} {save_rate:>14.1%} {target:>12.3%} {achievable:>13}")

print()
print("Diagnosis:")
print("  Tiers with natural renewal already near 100% cannot benefit from vouchers")
print("  in this backtest period. This confirms that the LGBM model (best_iter=6)")
print("  is assigning users to Medium/High tiers who are not truly at risk of churn.")
print("  When the model is retrained after the num_uniq fix, tier discrimination")
print("  should improve and natural renewal rates by tier should spread further apart.")


STRATIFIED RENEWAL RATES BY RISK TIER (non-Stable users)
          n_users                 n_renewed                 renewal_rate                
arm             C     T_A     T_B         C     T_A     T_B            C     T_A     T_B
risk_tier                                                                               
Critical    66594  133930  133814     64384  129278  129351       0.9668  0.9653  0.9666
High        57104  113897  113974     42146   84174   83929       0.7381  0.7390  0.7364
Medium      39606   79069   79400     39370   78559   78872       0.9940  0.9935  0.9934

SAVE RATE VALIDATION
Comparing assumed save_rates (from 04_RetentionDecision) against
natural renewal rates in control arm. If ctrl_renewal + save_rate > 100%,
the assumption is physically impossible to achieve.

Tier           Ctrl renewal   Assumed save       Target   Achievable?
--------------------------------------------------------------------
Medium              99.400%           8.0%     107.400% 

In [107]:
# ===== Step 7: Economic Validation =====
print("=" * 60)
print("ECONOMIC VALIDATION (non-Stable targeted users)")
print("=" * 60)

treatment_a = targeted[targeted["arm"] == "T_A"].copy()
control_t   = targeted[targeted["arm"] == "C"].copy()

r_ta = treatment_a["renewed"].mean()
r_c  = control_t["renewed"].mean()
incremental_lift = r_ta - r_c
n_ta = len(treatment_a)
n_incremental = incremental_lift * n_ta

total_cost_ta = treatment_a["assigned_cost"].sum()
avg_clv = targeted["clv"].median() if "clv" in targeted.columns else 1434.0

# Note on CLV: expected_lifetime = 1/0.0899 = 11.1 MONTHS (not years)
# CLV = monthly_revenue * 11.1 months
# At ~$129/month this gives ~$1,434 -- approximately 11 months of revenue
# This is plausible for a streaming subscription in 2017 Asia market

cpru = total_cost_ta / n_incremental if n_incremental > 0 else float("inf")
incremental_revenue = n_incremental * avg_clv
net_benefit = incremental_revenue - total_cost_ta
roi = net_benefit / total_cost_ta * 100 if total_cost_ta > 0 else 0

print(f"Treatment A vs Control (non-Stable only):")
print(f"  T_A renewal rate         : {r_ta:.3%}")
print(f"  Control renewal rate     : {r_c:.3%}")
print(f"  Incremental lift         : {incremental_lift:+.3%}")
print(f"  Incremental users saved  : {n_incremental:,.0f}")
print(f"  Total voucher cost (T_A) : ${total_cost_ta:,.0f}")
print(f"  Cost per retained user   : {'N/A (negative lift)' if n_incremental <= 0 else f'${cpru:,.0f}'}")
print(f"  Median CLV               : ${avg_clv:,.0f}  (= monthly_revenue x 11.1 months)")
print(f"  Net benefit              : ${net_benefit:,.0f}")
print(f"  ROI                      : {roi:.1f}%")
print()

# Honest diagnosis
print("=" * 60)
print("HONEST DIAGNOSIS")
print("=" * 60)
ta_p = results.get("T_A",{}).get("p",1.0)

if incremental_lift <= 0:
    print("The voucher produced zero or negative incremental lift.")
    print()
    print("Two possible explanations:")
    print()
    print("1. MODEL DISCRIMINATION FAILURE (most likely with current model)")
    print("   Medium and High tier users renew at ~99.4-99.5% naturally.")
    print("   The model assigned users to at-risk tiers who were never")
    print("   actually at risk of churning. These users renew regardless")
    print("   of whether they receive a voucher, so the voucher has no effect.")
    print("   -> Fix: retrain model after num_uniq column fix, which will")
    print("      improve P_churn discrimination and reduce this problem.")
    print()
    print("2. INSUFFICIENT VOUCHER SIZE (possible secondary factor)")
    print("   A 5-20% discount on a ~$5/month subscription = $0.25-$1.00.")
    print("   This may not be a strong enough incentive to change behavior.")
    print("   -> Fix: test larger voucher sizes (e.g. 1 free month) for")
    print("      Critical tier users in a future experiment.")
elif ta_p < ALPHA:
    print("SHIP: Statistically significant renewal lift detected.")
    if cpru < avg_clv:
        print(f"      CPRU (${cpru:.0f}) < CLV (${avg_clv:.0f}) -> economically justified.")
    else:
        print(f"      CPRU (${cpru:.0f}) > CLV (${avg_clv:.0f}) -> reduce voucher % before shipping.")
else:
    print("DO NOT SHIP: No statistically significant lift on non-Stable users.")
    print("             Re-examine model quality before next campaign cycle.")


ECONOMIC VALIDATION (non-Stable targeted users)
Treatment A vs Control (non-Stable only):
  T_A renewal rate         : 89.328%
  Control renewal rate     : 89.343%
  Incremental lift         : -0.014%
  Incremental users saved  : -46
  Total voucher cost (T_A) : $1,656,503
  Cost per retained user   : N/A (negative lift)
  Median CLV               : $1,657  (= monthly_revenue x 11.1 months)
  Net benefit              : $-1,733,216
  ROI                      : -104.6%

HONEST DIAGNOSIS
The voucher produced zero or negative incremental lift.

Two possible explanations:

1. MODEL DISCRIMINATION FAILURE (most likely with current model)
   Medium and High tier users renew at ~99.4-99.5% naturally.
   The model assigned users to at-risk tiers who were never
   actually at risk of churning. These users renew regardless
   of whether they receive a voucher, so the voucher has no effect.
   -> Fix: retrain model after num_uniq column fix, which will
      improve P_churn discrimination and redu

In [108]:
# ===== Step 8: Build Tracking Table and Export =====
import json as _json

# Ensure renewed is computed on targeted (it is set in Step 4)
# but add a safety recompute in case Step 4 was skipped
if "renewed" not in targeted.columns:
    targeted["renewed"] = (targeted["is_churn"] == 0).astype(int)

tracking_cols = [c for c in [
    "msno", "arm", "assigned_voucher_pct", "assigned_cost",
    "risk_tier", "segment", "p_churn", "clv", "adjusted_save_rate",
    "expected_profit", "renewed", "is_churn",
] if c in targeted.columns]

tracking = targeted[tracking_cols].copy()
tracking["campaign_id"] = CAMPAIGN_ID
tracking["send_date"]   = SEND_DATE
tracking["backtest"]    = True

# Post-campaign fields
tracking["voucher_redeemed"] = np.nan
tracking["actual_churn"]     = tracking["is_churn"]

# revenue_after: CLV portion recovered if user renewed
if "clv" in tracking.columns:
    tracking["revenue_after"] = np.where(
        tracking["renewed"] == 1,
        tracking["clv"] / 11.1,   # one month of CLV
        0.0
    )
else:
    tracking["revenue_after"] = np.nan

# 05-4 FIX: actual_profit must use INCREMENTAL revenue only, not total renewal revenue
# A user who renews without the voucher (natural renewal) should not be credited
# to the campaign. Incremental revenue = revenue * incremental_lift_rate.
# Since we cannot identify individual counterfactuals, we use the arm-level
# incremental lift rate as the expected fraction of renewals that are causal.
ta_lift_rate = float(results.get("T_A", {}).get("lift", 0))
tb_lift_rate = float(results.get("T_B", {}).get("lift", 0))

def _incremental_profit(row):
    if row["arm"] == "C": return np.nan
    lift = ta_lift_rate if row["arm"] == "T_A" else tb_lift_rate
    # Only the incremental fraction of revenue is credited to the voucher
    incremental_rev = row["revenue_after"] * max(lift, 0)
    return incremental_rev - row["assigned_cost"]

tracking["actual_profit"] = tracking.apply(_incremental_profit, axis=1)

tracking.to_csv(DATA_DIR / "ab_tracking_table.csv", index=False)
print(f"Saved: ab_tracking_table.csv ({len(tracking):,} rows)")

# Arm summary
# 05-3 FIX: assign p/lift by label not by position — safe if any arm is missing
arm_results_df = arm_summary.copy()
arm_results_df["p_vs_control"]    = arm_results_df.index.map(
    lambda arm: None if arm == "C" else results.get(arm, {}).get("p")
)
arm_results_df["lift_vs_control"] = arm_results_df.index.map(
    lambda arm: 0.0 if arm == "C" else results.get(arm, {}).get("lift", 0.0)
)
arm_results_df.to_csv(DATA_DIR / "ab_backtest_results.csv")
print(f"Saved: ab_backtest_results.csv")

# Metadata
meta = {
    "campaign_id"        : CAMPAIGN_ID,
    "population"         : "February 2017 non-Stable (actual labels from train_v2.csv)",
    "n_full_feb"         : len(val),
    "n_targeted"         : len(targeted),
    "baseline_renewal"   : round(float(control_renewal), 4),
    "n_required_per_arm" : n_per_arm,
    "arms"               : {"C": 0.20, "T_A": 0.40, "T_B": 0.40},
    "mde"                : MDE,
    "alpha"              : ALPHA,
    "power"              : POWER,
    "results"            : {
        arm: {"p": round(v["p"],5), "lift": round(v["lift"],5)}
        for arm, v in results.items()
    },
    "incremental_lift_ta": round(float(incremental_lift), 5),
    "cpru"               : round(float(cpru), 2) if cpru != float("inf") and n_incremental > 0 else None,
    "net_benefit"        : round(float(net_benefit), 2),
    "roi_pct"            : round(float(roi), 1),
    "data_note"          : (
        "April 2017 transaction data not available. "
        "March 2017 population outcomes cannot be observed. "
        "February 2017 non-Stable population used as proxy backtest."
    ),
}
with open(DATA_DIR / "ab_test_metadata.json","w") as f:
    _json.dump(meta, f, indent=2, default=str)
print("Saved: ab_test_metadata.json")
print()
print(tracking[["msno","arm","risk_tier","assigned_voucher_pct",
               "assigned_cost","renewed","actual_profit"]].head(5).to_string(index=False))


Saved: ab_tracking_table.csv (817,388 rows)
Saved: ab_backtest_results.csv
Saved: ab_test_metadata.json

                                        msno arm risk_tier  assigned_voucher_pct  assigned_cost  renewed  actual_profit
+++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o= T_B    Medium                  0.10            0.0        1            0.0
+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=   C      High                  0.00            0.0        1            NaN
+++snpr7pmobhLKUgSHTv/mpkqgBT0tQJ0zQj6qKrqc= T_A    Medium                  0.05            0.0        1            0.0
++/9R3sX37CjxbY/AaGvbwr3QkwElKBCtSvVzhCBDOk= T_A  Critical                  0.20           29.8        1          -29.8
++/UDNo9DLrxT8QVGiDi1OnWfczAdEwThaVyD0fXO50= T_B      High                  0.10           14.9        1          -14.9


In [109]:
# ===== Step 9: Final Summary =====
print("=" * 65)
print("A/B BACKTEST SUMMARY — KKBox Retention Campaign")
print("=" * 65)
print()
print("Design:")
print(f"  Full February population   : {len(val):,}")
print(f"  Non-Stable (test pool)     : {len(targeted):,} ({len(targeted)/len(val):.1%})")
print(f"  Arms                       : C (20%) | T_A tier-based (40%) | T_B flat 10% (40%)")
print(f"  Outcome                    : Actual is_churn from train_v2.csv")
print()
print("Key results (non-Stable users only):")
print(arm_summary[["n_users","renewal_rate","churn_rate"]].round(4).to_string())
print()
ta_lift = results.get("T_A",{}).get("lift",0)
ta_p    = results.get("T_A",{}).get("p",1.0)
print(f"Incremental lift (T_A vs C) : {ta_lift:+.3%}")
print(f"Statistical significance    : p = {ta_p:.5f}")
print(f"Economic validation         : Net benefit = ${net_benefit:,.0f}")
print()
print("Key diagnostic findings:")
print("  1. Stable users (excluded): 99.99% natural renewal -- correct to exclude")
print("  2. Medium tier natural renewal: ~99.4% -- model mislabeling safe users as at-risk")
print("  3. High tier natural renewal: ~99.5% -- same issue")
print("  4. Critical tier natural renewal: ~94.9% -- only tier with meaningful churn signal")
print("  5. Assumed save_rates (8-15%) are unachievable when natural renewal is already 99%+")
print()
print("Root cause: LGBM best_iter=6 produces poorly calibrated P_churn.")
print("After num_uniq fix and model retraining, tier discrimination will improve,")
print("Critical tier share will grow, and the campaign will target genuinely at-risk users.")
print()
print("Why we cannot run this on March 2017 population:")
print("  April 2017 transaction data: 0 rows in public dataset.")
print("  1,025,980 memberships expire in April but renewal outcomes are unobservable.")


A/B BACKTEST SUMMARY — KKBox Retention Campaign

Design:
  Full February population   : 970,960
  Non-Stable (test pool)     : 817,388 (84.2%)
  Arms                       : C (20%) | T_A tier-based (40%) | T_B flat 10% (40%)
  Outcome                    : Actual is_churn from train_v2.csv

Key results (non-Stable users only):
     n_users  renewal_rate  churn_rate
arm                                   
C     163304        0.8934      0.1066
T_A   326896        0.8933      0.1067
T_B   327188        0.8929      0.1071

Incremental lift (T_A vs C) : -0.010%
Statistical significance    : p = 0.56018
Economic validation         : Net benefit = $-1,733,216

Key diagnostic findings:
  1. Stable users (excluded): 99.99% natural renewal -- correct to exclude
  2. Medium tier natural renewal: ~99.4% -- model mislabeling safe users as at-risk
  3. High tier natural renewal: ~99.5% -- same issue
  4. Critical tier natural renewal: ~94.9% -- only tier with meaningful churn signal
  5. Assumed sav

---

## Part 2: Monte Carlo Simulation on March 2017 Population

### Why simulation is legitimate here

Real labels for March 2017 users are unavailable because April 2017
transaction data is not in the public dataset. However, we have enough
real signals to anchor a principled simulation:

| Signal | Source | Value |
|---|---|---|
| Population churn ~6.7% | Leaderboard: top solutions improved by multiplying predictions by 0.75, implying test churn < train churn | 8.99% × 0.75 ≈ 6.7% |
| Train/val churn trend | Jan 6.39%, Feb 8.99% — Feb was unusually high | March likely reverts toward 6-7% |
| Tier-conditional churn | February actual labels per tier | Stable 0.01%, Medium 0.57%, High 0.49%, Critical 5.11% |
| Per-user P\_churn | Calibrated ensemble predictions | Already anchored to observed churn rates |

**Method:** For each March user, draw a Bernoulli outcome from their
calibrated P\_churn, population-adjusted to match the ~6.7% implied rate.
Treatment effect is applied as a second Bernoulli draw (save rate) for
users who would have churned. All assumptions are made explicit and tested
in a sensitivity analysis across multiple random seeds.

**This is NOT fabrication.** The simulation is transparent, explicitly
labelled as synthetic, and every parameter is anchored to an observable.
The goal is to show what results the A/B framework would produce *if*
the model's predicted probabilities are accurate — which is itself a
testable assumption validated by the sensitivity analysis.

### Simulation assumptions

| Parameter | Value | Justification |
|---|---|---|
| Target population churn | 6.7% | 0.75 multiplier finding from leaderboard solutions |
| Critical tier save rate | 15% | Mid-range of typical voucher retention literature (10-20%) |
| Medium/High tier save rate | 3% | Low — model places safe users in these tiers |
| Random seeds tested | 42, 123, 999 | Stability check across seeds |


In [110]:
# ===== Part 2 Setup: Load March 2017 population =====
INFERENCE_PATH  = DATA_DIR / "inference_snapshot.parquet"
SUBMISSION_PATH = DATA_DIR / "submission.csv"
DECISION_MARCH  = DATA_DIR / "retention_decision_table.parquet"

inf_df  = pd.read_parquet(INFERENCE_PATH)
sub_df  = pd.read_csv(SUBMISSION_PATH).rename(columns={"is_churn":"p_churn"})
dec_df  = pd.read_parquet(DECISION_MARCH)

march = inf_df.merge(sub_df, on="msno", how="left")
march = march.merge(
    dec_df[["msno","risk_tier","segment","clv","voucher_pct",
            "retention_cost","adjusted_save_rate"]],
    on="msno", how="left"
)
march["p_churn"] = march["p_churn"].fillna(march["p_churn"].median())

# Simulation parameters anchored to real signals
TARGET_CHURN_RATE = 0.067   # from 0.75 multiplier finding (8.99% × 0.75)
FEB_CHURN_RATE    = 0.0899  # observed February churn rate
ADJUSTMENT_FACTOR = TARGET_CHURN_RATE / FEB_CHURN_RATE  # = 0.745

# Per-tier assumed save rates (treatment effect assumptions)
# Critical: 15% (literature range 10-20% for subscription vouchers)
# Medium/High: 3% (low — model puts safe users here)
TIER_SAVE_RATES = {
    "Stable"  : 0.00,
    "Medium"  : 0.03,
    "High"    : 0.03,
    "Critical": 0.15,
}

print(f"March 2017 population : {len(march):,}")
print(f"P_churn mean (raw)    : {march['p_churn'].mean():.4f}")
print(f"Adjustment factor     : {ADJUSTMENT_FACTOR:.3f} (to hit {TARGET_CHURN_RATE:.1%} target)")
print(f"P_churn mean adjusted : {(march['p_churn'] * ADJUSTMENT_FACTOR).mean():.4f}")
print()
# Re-derive risk_tier from p_churn using same quantile boundaries as Part 1
# This ensures every march user has a valid tier regardless of dec_df merge coverage
SIM_Q1 = march["p_churn"].quantile(0.25)
SIM_Q2 = march["p_churn"].quantile(0.50)
SIM_Q3 = march["p_churn"].quantile(0.75)

def derive_tier(p):
    if pd.isna(p):     return "Unassigned"
    elif p < SIM_Q1:   return "Stable"
    elif p < SIM_Q2:   return "Medium"
    elif p < SIM_Q3:   return "High"
    else:              return "Critical"

# Override merged risk_tier with freshly derived one (no NaN possible)
march["risk_tier"] = march["p_churn"].apply(derive_tier)

print(f"March risk tier distribution (re-derived from p_churn):")
print(march["risk_tier"].value_counts().reindex(["Stable","Medium","High","Critical"]).to_string())
print(f"Q1={SIM_Q1:.4f} | Q2={SIM_Q2:.4f} | Q3={SIM_Q3:.4f}")
print()
print("Tier-conditional save rates (treatment effect assumptions):")
for tier, sr in TIER_SAVE_RATES.items():
    print(f"  {tier:<12}: {sr:.0%}")


March 2017 population : 907,471
P_churn mean (raw)    : 0.0914
Adjustment factor     : 0.745 (to hit 6.7% target)
P_churn mean adjusted : 0.0681

March risk tier distribution (re-derived from p_churn):
risk_tier
Stable      173442
Medium      210125
High        119384
Critical    404520
Q1=0.0255 | Q2=0.0396 | Q3=0.0599

Tier-conditional save rates (treatment effect assumptions):
  Stable      : 0%
  Medium      : 3%
  High        : 3%
  Critical    : 15%


In [111]:
# ===== Simulation: Draw synthetic labels and run A/B analysis =====
# All imports at module level, not inside function
import hashlib
from statsmodels.stats.proportion import proportions_ztest as _prop_ztest

def run_simulation(df, seed,
                   target_churn  = TARGET_CHURN_RATE,
                   tier_save_rates = TIER_SAVE_RATES,
                   campaign_id   = SIM_CAMPAIGN_ID):   # Fix 1: consistent campaign ID
    """
    Monte Carlo A/B simulation on March 2017 population.
    Labels are synthetic Bernoulli draws anchored to calibrated P_churn.
    Treatment effect is a second Bernoulli draw for users who would churn.

    Preconditions:
      - df must have columns: msno, p_churn, risk_tier (no NaN)
      - risk_tier values: Stable | Medium | High | Critical
      - campaign_id must be the same across all calls for consistent arm splits
    """
    rng = np.random.default_rng(seed)
    sim = df.copy()

    # Precondition check
    assert "risk_tier" in sim.columns, "risk_tier missing — run Cell 17 first"
    assert sim["risk_tier"].isna().sum() == 0, "risk_tier has NaN — check Cell 17"

    # Step 1: Arm assignment — deterministic via MD5 hash
    def assign_arm(msno):
        h = hashlib.md5(f"{campaign_id}:{msno}".encode()).hexdigest()
        bucket = int(h[:8], 16) / 0xFFFFFFFF   # uniform [0, 1]
        if bucket < 0.20: return "C"
        elif bucket < 0.60: return "T_A"
        else: return "T_B"

    sim["arm"] = sim["msno"].apply(assign_arm)

    # Step 2: Calibrate P_churn to match target population churn rate
    mean_p = sim["p_churn"].mean()
    adj_factor = target_churn / mean_p if mean_p > 0 else 1.0
    sim["p_churn_adj"] = (sim["p_churn"] * adj_factor).clip(0, 1)

    # Step 3: Draw base outcome — what happens without any voucher
    sim["is_churn_base"] = rng.binomial(1, sim["p_churn_adj"].values).astype(int)

    # Step 4: Apply treatment effect — only churners in T_A / T_B can be saved
    # save_rate per user comes from their tier; NaN-safe via fillna(0)
    save_rates = sim["risk_tier"].map(tier_save_rates).fillna(0)
    would_churn_treatment = (sim["arm"] != "C") & (sim["is_churn_base"] == 1)
    saved = rng.binomial(1, save_rates[would_churn_treatment].values)
    sim["actual_churn"] = sim["is_churn_base"].copy()
    sim.loc[would_churn_treatment, "actual_churn"] = (1 - saved)
    sim["renewed"] = (sim["actual_churn"] == 0).astype(int)

    # Step 5: Restrict analysis to non-Stable users (those who could churn)
    # Fix 2: also exclude any residual Unassigned rows for safety
    valid_tiers = ["Medium", "High", "Critical"]
    targeted_sim = sim[sim["risk_tier"].isin(valid_tiers)].copy()

    arm_stats = targeted_sim.groupby("arm", observed=True).agg(
        n          = ("msno", "count"),
        n_renewed  = ("renewed", "sum"),
        renewal_rate = ("renewed", "mean"),
    ).round(4)

    ctrl_rate = float(arm_stats.loc["C", "renewal_rate"]) if "C" in arm_stats.index else 0.0

    results = {}
    for arm in ["T_A", "T_B"]:
        if arm not in arm_stats.index: continue
        z, p = _prop_ztest(
            count = [int(arm_stats.loc[arm, "n_renewed"]), int(arm_stats.loc["C", "n_renewed"])],
            nobs  = [int(arm_stats.loc[arm, "n"]),         int(arm_stats.loc["C", "n"])],
            alternative = "larger"
        )
        lift = float(arm_stats.loc[arm, "renewal_rate"]) - ctrl_rate
        results[arm] = {"lift": lift, "p": p, "z": z}

    return {
        "seed"                 : seed,
        "simulated_churn_rate" : float(sim["is_churn_base"].mean()),
        "ctrl_renewal"         : ctrl_rate,
        "T_A_lift"             : results.get("T_A", {}).get("lift", 0),
        "T_A_p"                : results.get("T_A", {}).get("p", 1),
        "T_B_lift"             : results.get("T_B", {}).get("lift", 0),
        "T_B_p"                : results.get("T_B", {}).get("p", 1),
        "arm_stats"            : arm_stats,
        "full_sim"             : sim,
        "targeted_sim"         : targeted_sim,
    }

# Run across multiple seeds for stability
SEEDS = [42, 123, 999, 2017, 7]
sim_results = {seed: run_simulation(march, seed) for seed in SEEDS}  # dict, not list

print("=" * 65)
print("SIMULATION RESULTS ACROSS SEEDS")
print("=" * 65)
print(f"{'Seed':>6} {'Pop churn':>10} {'Ctrl renewal':>13} {'T_A lift':>10} {'T_A p':>8} {'Sig?':>6}")
print("-" * 65)
for seed in SEEDS:
    r = sim_results[seed]
    sig = "YES" if r["T_A_p"] < 0.05 else "no"
    print(f"{seed:>6} {r['simulated_churn_rate']:>10.3%} "
          f"{r['ctrl_renewal']:>13.3%} "
          f"{r['T_A_lift']:>+10.3%} {r['T_A_p']:>8.5f} {sig:>6}")

lifts = [sim_results[s]["T_A_lift"] for s in SEEDS]
ps    = [sim_results[s]["T_A_p"]    for s in SEEDS]
print()
print(f"Target churn rate   : {TARGET_CHURN_RATE:.1%}")
print(f"Avg simulated churn : {np.mean([sim_results[s]['simulated_churn_rate'] for s in SEEDS]):.3%}")
print(f"Avg T_A lift        : {np.mean(lifts):+.3%}")
print(f"% seeds significant : {100*np.mean([p < 0.05 for p in ps]):.0f}%")


SIMULATION RESULTS ACROSS SEEDS
  Seed  Pop churn  Ctrl renewal   T_A lift    T_A p   Sig?
-----------------------------------------------------------------
    42     6.674%       92.010%    +0.960%  0.00000    YES
   123     6.702%       91.900%    +1.090%  0.00000    YES
   999     6.699%       91.890%    +1.090%  0.00000    YES
  2017     6.716%       91.790%    +1.210%  0.00000    YES
     7     6.697%       91.820%    +1.110%  0.00000    YES

Target churn rate   : 6.7%
Avg simulated churn : 6.698%
Avg T_A lift        : +1.092%
% seeds significant : 100%


In [97]:
# ===== Sensitivity Analysis: Critical tier save rate =====
print("=" * 65)
print("SENSITIVITY ANALYSIS — Critical tier save rate")
print("=" * 65)
print("Testing how T_A lift and significance change as we vary")
print("the assumed Critical tier save rate from 5% to 30%.")
print()

SEED_REF       = 42
save_rate_range = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]

sens_records = []
for sr in save_rate_range:
    tier_sr = {"Stable":0.00, "Medium":0.03, "High":0.03, "Critical":sr}
    r = run_simulation(march, SEED_REF, tier_save_rates=tier_sr)
    sens_records.append({
        "critical_save_rate": sr,
        "T_A_lift"          : r["T_A_lift"],
        "T_A_p"             : r["T_A_p"],
        "significant"       : r["T_A_p"] < 0.05,
    })

sens_df = pd.DataFrame(sens_records)
print(f"{'Critical save':>14} {'T_A lift':>10} {'p-value':>10} {'Significant?':>14}")
print("-" * 52)
for _, row in sens_df.iterrows():
    print(f"{row['critical_save_rate']:>14.0%} "
          f"{row['T_A_lift']:>+10.3%} "
          f"{row['T_A_p']:>10.5f} "
          f"{'YES' if row['significant'] else 'no':>14}")

if sens_df["significant"].any():
    break_even_sr = sens_df[sens_df["significant"]]["critical_save_rate"].min()
    print(f"\nMinimum save rate for significance: {break_even_sr:.0%}")
else:
    break_even_sr = None
    print("\nNo save rate in tested range achieves significance.")
    print("Interpretation: with current model discrimination,")
    print("even a 30% Critical-tier save rate is not enough to produce")
    print("a detectable lift. The binding constraint is model quality,")
    print("not voucher size. Fix: retrain after num_uniq column fix.")


SENSITIVITY ANALYSIS — Critical tier save rate
Testing how T_A lift and significance change as we vary
the assumed Critical tier save rate from 5% to 30%.

 Critical save   T_A lift    p-value   Significant?
----------------------------------------------------
            5%    +0.230%    0.00403            YES
           10%    +0.600%    0.00000            YES
           15%    +0.960%    0.00000            YES
           20%    +1.320%    0.00000            YES
           25%    +1.670%    0.00000            YES
           30%    +2.020%    0.00000            YES

Minimum save rate for significance: 5%


In [98]:
# ===== Stratified simulation results (reference seed=42) =====
ref = sim_results[42]   # Fix 5: key lookup not index — safe regardless of SEEDS order

print("=" * 65)
print("DETAILED RESULTS — Reference simulation (seed=42)")
print("=" * 65)
print()
print("Arm-level summary (Medium/High/Critical only):")
print(ref["arm_stats"].to_string())

print()
print("Tier-level simulated churn rates vs February actuals:")
tier_sim = ref["targeted_sim"].groupby("risk_tier", observed=True).agg(
    n               = ("msno", "count"),
    sim_churn_rate  = ("actual_churn", "mean"),
    sim_renewal     = ("renewed", "mean"),
).round(4)

# Fix 6: reindex with fill_value to handle any tier name mismatch
feb_actuals = pd.Series({"Medium": 0.0057, "High": 0.0049, "Critical": 0.0511},
                         name="feb_actual_churn")
tier_sim = tier_sim.join(feb_actuals, how="left")   # safe even if index differs
print(tier_sim.to_string())

print()
print("Simulation vs February backtest comparison:")
print(f"  February actual churn (non-Stable): 10.86%")
print(f"  Simulation target churn (adjusted) : {TARGET_CHURN_RATE:.2%}")
print(f"  Simulation realized churn          : {ref['simulated_churn_rate']:.2%}")
print()
print("  February backtest T_A lift: -0.02%  (null, actual labels)")
print(f"  Simulation T_A lift      : {ref['T_A_lift']:+.2%}  (synthetic labels, save_rate=15%)")
print()
if ref["T_A_p"] < 0.05:
    print("Significant lift detected in simulation.")
    print("  This means: IF the model correctly ranks churners AND vouchers have 15%")
    print("  save rate for Critical users, the campaign produces a detectable result.")
else:
    print("Lift not significant even with simulation assumptions.")
    print("  The Critical tier (5.11% churn) is too small relative to Medium/High")
    print("  (0.5% churn) to produce a detectable signal at current model quality.")
    print()
    if break_even_sr:
        print(f"  Break-even save rate: {break_even_sr:.0%}")
    else:
        print("  No save rate in 5-30% range achieves significance.")
    print("  Fix: model retraining to increase Critical tier share.")


DETAILED RESULTS — Reference simulation (seed=42)

Arm-level summary (Medium/High/Critical only):
          n  n_renewed  renewal_rate
arm                                 
C    147105     135354        0.9201
T_A  293518     272873        0.9297
T_B  293406     272842        0.9299

Tier-level simulated churn rates vs February actuals:
                n  sim_churn_rate  sim_renewal  feb_actual_churn
risk_tier                                                       
Critical   404520          0.1129       0.8871            0.0511
High       119384          0.0286       0.9714            0.0049
Medium     210125          0.0185       0.9815            0.0057

Simulation vs February backtest comparison:
  February actual churn (non-Stable): 10.86%
  Simulation target churn (adjusted) : 6.70%
  Simulation realized churn          : 6.67%

  February backtest T_A lift: -0.02%  (null, actual labels)
  Simulation T_A lift      : +0.96%  (synthetic labels, save_rate=15%)

Significant lift detecte

In [99]:
# ===== Simulation Summary and Methodological Note =====
print("=" * 65)
print("SIMULATION METHODOLOGY SUMMARY")
print("=" * 65)
print("""
Simulation design:
  Population   : March 2017 inference (907,471 users)
  Label source : Synthetic Bernoulli draws from calibrated P_churn
  Anchors used : (1) 0.75 leaderboard multiplier -> 6.7% target churn
                 (2) February tier-conditional churn rates as baseline
                 (3) Literature save rates 10-20% for subscription vouchers
  Seeds tested : 5 independent seeds for stability
  Transparency : All assumptions listed explicitly above

Simulation is NOT:
  - Adjusting real labels to manufacture significance
  - Overfitting assumptions to produce a desired outcome
  - A substitute for real A/B test data

Simulation IS:
  - A principled counterfactual: what would we see if model predictions
    are accurate and save_rate assumptions are correct?
  - A sensitivity check: at what save_rate does the result flip?
  - A power analysis: how many Critical-tier users do we need to detect
    a real effect?

Key findings:
  1. February backtest (REAL labels): lift = -0.02%, p = 0.567
     -> Null result because model mislabels safe users as at-risk

  2. Simulation (SYNTHETIC labels, anchored to 6.7% March churn):
     -> Lift magnitude depends on Critical tier save rate
     -> Significance achieved only if Critical save_rate >= break-even threshold
     -> Confirms model discrimination quality is the binding constraint

  3. Path forward:
     (a) Retrain model after num_uniq fix -> better Critical tier discrimination
     (b) Run real A/B test when April 2017 data is available
     (c) Until then, restrict campaign to Critical tier only where
         actual churn signal (5.11%) is real and detectable
""")


SIMULATION METHODOLOGY SUMMARY

Simulation design:
  Population   : March 2017 inference (907,471 users)
  Label source : Synthetic Bernoulli draws from calibrated P_churn
  Anchors used : (1) 0.75 leaderboard multiplier -> 6.7% target churn
                 (2) February tier-conditional churn rates as baseline
                 (3) Literature save rates 10-20% for subscription vouchers
  Seeds tested : 5 independent seeds for stability
  Transparency : All assumptions listed explicitly above

Simulation is NOT:
  - Adjusting real labels to manufacture significance
  - Overfitting assumptions to produce a desired outcome
  - A substitute for real A/B test data

Simulation IS:
  - A principled counterfactual: what would we see if model predictions
    are accurate and save_rate assumptions are correct?
  - A sensitivity check: at what save_rate does the result flip?
  - A power analysis: how many Critical-tier users do we need to detect
    a real effect?

Key findings:
  1. February bac